In [8]:
import pandas as pd
import boto3
import io

# Configurações iniciais
arquivo_original = "consumo-energia.csv"
bucket = "desafio-da-sprint04"
caminho_s3 = "consumo_energia_limpo.csv"
perfil_aws = "Yanmaiscedo"

# Leitura do arquivo e limpeza
df = pd.read_csv(arquivo_original)
linhas_antes = df.shape[0]

df.columns = df.columns.str.strip()

df['media_consumo_mes_2018_2019'] = pd.to_numeric(df['media_consumo_mes_2018_2019'], errors='coerce')
df['consumo_mes_referencia'] = pd.to_numeric(df['consumo_mes_referencia'], errors='coerce')

df['mes_ano'] = df['mes_ano'].astype(str).str.zfill(6)
df['mes'] = df['mes_ano'].str.slice(0, len(df['mes_ano'][0]) - 4).astype(int)
df['ano'] = df['mes_ano'].str[-4:].astype(int)

df['mes_ano_corrigido'] = pd.to_datetime(
    df['ano'].astype(str) + '-' + df['mes'].astype(str).str.zfill(2),
    format='%Y-%m',
    errors='coerce'
)

df_limpo = df.dropna(subset=['media_consumo_mes_2018_2019', 'consumo_mes_referencia', 'orgao'])

colunas_relevantes = [
    'mes_ano_corrigido',
    'mes_ano',
    'orgao',
    'sigla_orgao',
    'media_consumo_mes_2018_2019',
    'consumo_mes_referencia',
    'justificativa_meta',
    'observacao'
]
df_limpo = df_limpo[colunas_relevantes]
linhas_depois = df_limpo.shape[0]

# Envio para o Bucket
session = boto3.Session(profile_name=perfil_aws)
s3 = session.client("s3")

buffer = io.StringIO()
df_limpo.to_csv(buffer, index=False)
buffer.seek(0)

s3.put_object(Body=buffer.getvalue(), Bucket=bucket, Key=caminho_s3)

print("Número de linhas antes da limpeza:", linhas_antes)
print("Número de linhas após a limpeza:", linhas_depois)
print(f"\nArquivo limpo enviado com sucesso para s3://{bucket}/{caminho_s3}")

Número de linhas antes da limpeza: 803
Número de linhas após a limpeza: 803

Arquivo limpo enviado com sucesso para s3://desafio-da-sprint04/consumo_energia_limpo.csv


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import io

# Configurações
arquivo_csv = "consumo_energia_limpo.csv"
bucket = "desafio-da-sprint04"
pasta_s3 = "analises-etapa1/"
perfil_aws = "Yanmaiscedo"

# Sessão AWS
session = boto3.Session(profile_name=perfil_aws)
s3 = session.client('s3')

# Leitura do CSV direto do Bucker
obj = s3.get_object(Bucket=bucket, Key=f"{arquivo_csv}")
df = pd.read_csv(io.BytesIO(obj["Body"].read()), parse_dates=["mes_ano_corrigido"])

# Analise 1: Economia de Energia
df['variacao_percentual'] = ((df['consumo_mes_referencia'] - df['media_consumo_mes_2018_2019']) / df['media_consumo_mes_2018_2019']) * 100
economias = df.groupby('sigla_orgao')['variacao_percentual'].mean().sort_values()
top_5_economias = economias.head(5)

with open("analise1.txt", "w") as f:
    f.write("Top 5 órgãos que mais economizaram energia:\n")
    f.write(top_5_economias.to_string())

# Analise 2: Aumento de consumo de Energia
top_5_desperdicio = economias.tail(5)
with open("analise2.txt", "w") as f:
    f.write("Top 5 órgãos que mais aumentaram o consumo de energia:\n")
    f.write(top_5_desperdicio.to_string())

# Analise 3: Maiores Consumidores de Energia
orgao_consumo = df.groupby("sigla_orgao")["consumo_mes_referencia"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=orgao_consumo.values,
    y=orgao_consumo.index,
    hue=orgao_consumo.index,
    palette='viridis',
    legend=False
)
plt.title("Top 10 órgãos com maior consumo total")
plt.xlabel("Consumo Total (kWh)")
plt.tight_layout()
plt.savefig("analise3.png")
plt.close()

# Analise 4: Justificativas dos que não atingiram a meta
df["atingiu_meta"] = df["consumo_mes_referencia"] < df["media_consumo_mes_2018_2019"]
df["atingiu_meta"] = df["atingiu_meta"].map({True: "SIM", False: "NÃO"})

nao_cumpriram = df[df["atingiu_meta"] == "NÃO"]
analise4_df = nao_cumpriram[["orgao", "media_consumo_mes_2018_2019", "consumo_mes_referencia", "justificativa_meta"]].head()

texto_analise4 = "ANÁLISE 4: Justificativas dos órgãos que não atingiram a meta (Top 5)\n\n"
for idx, row in analise4_df.iterrows():
    texto_analise4 += f"Órgão: {row['orgao']}\n"
    texto_analise4 += f"Consumo médio histórico: {row['media_consumo_mes_2018_2019']} kWh\n"
    texto_analise4 += f"Consumo atual: {row['consumo_mes_referencia']} kWh\n"
    texto_analise4 += f"Justificativa: {row['justificativa_meta']}\n"
    texto_analise4 += "-" * 40 + "\n"

with open("analise4.txt", "w", encoding="utf-8") as f:
    f.write(texto_analise4)

# Envio dos resultados para a pasta analises-etapa1/ no Bucket
arquivos = [
    "analise1.txt",
    "analise2.txt",
    "analise3.png",
    "analise4.txt"
]

for arquivo in arquivos:
    s3.upload_file(arquivo, bucket, f"{pasta_s3}{arquivo}")
    print(f"{arquivo} enviado para o bucket '{bucket}' na pasta '{pasta_s3}'")

analise1.txt enviado para o bucket 'desafio-da-sprint04' na pasta 'analises-etapa1/'
analise2.txt enviado para o bucket 'desafio-da-sprint04' na pasta 'analises-etapa1/'
analise3.png enviado para o bucket 'desafio-da-sprint04' na pasta 'analises-etapa1/'
analise4.txt enviado para o bucket 'desafio-da-sprint04' na pasta 'analises-etapa1/'
